# Lesson 01 - 人工智能代理介绍

欢迎来到**AI代理初学者**课程的第一课！

**AI代理**是一个使用大型语言模型（LLM）作为其推理引擎的程序，能够在现实世界中采取*行动*——调用API、查询数据库或运行代码——以代表用户实现目标。

在本笔记本中，您将构建第一个代理：一个推荐度假目的地的**旅行代理**。在此过程中，您将学习如何：

1. 使用**Microsoft代理框架**连接到Azure AI Foundry代理服务。
2. 给代理一个**工具**——它可以调用的普通Python函数。
3. 运行代理并检查其响应。
4. 按令牌流式传输代理的响应。


## 设置

在运行此笔记本之前，请确保您已完成以下操作：

1. **拥有一个 Azure AI Foundry 项目** 并且部署了聊天模型（例如 `gpt-4o-mini`）。
2. **已使用 Azure CLI 登录** — 在终端运行 `az login`。
3. **设置所需的环境变量：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 项目端点。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您已部署模型的名称。

下面的代码单元将安装您需要的 Python 包。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 创建您的第一个代理

代理需要两样东西：

- **指令**，告诉它*它是谁*以及*如何行为*（系统提示）。
- **工具** — 使用 `@tool` 装饰的 Python 函数，代理可以调用这些函数来获取信息或执行操作。

下面我们定义了一个简单的工具，它返回一个流行的度假目的地列表。当用户请求旅行推荐时，代理将使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 流式响应

为了获得更互动的体验，您可以**流式传输**代理的响应。代理不会等待完整回复，而是随着生成逐步输出文本块。这在聊天界面中特别有用，因为您希望实时显示输出内容。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

在本课中，您学习了如何：

- **创建提供程序**，通过 `AzureAIProjectAgentProvider` 连接到 Azure AI Foundry Agent 服务。
- **使用 `@tool` 装饰器定义工具**，以便代理可以调用您的 Python 函数。
- **运行代理**，传入用户消息并打印其响应。
- **流式响应**，实现实时输出。

在下一课中，我们将更深入地探讨代理框架，学习如何为代理提供更强大的工具和多步骤推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：  
本文件由人工智能翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译。虽然我们力求准确，但请注意自动翻译可能包含错误或不准确之处。原始文件的母语版本应被视为权威来源。对于重要信息，建议使用专业人工翻译。对于因使用本翻译而引起的任何误解或错误解释，我们不承担任何责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
